# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [195]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split



import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [196]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [197]:
spaceship.shape

(8693, 14)

**Check for data types**

In [198]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [199]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [200]:
spaceship.dropna(inplace=True)

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [201]:
spaceship['Cabin'] = spaceship["Cabin"].str.split("/").str[0]
spaceship

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


- Drop PassengerId and Name

In [202]:
spaceship.drop(columns=['Name', 'PassengerId'], inplace=True)

In [203]:
spaceship['HomePlanet'].value_counts()

HomePlanet
Earth     3566
Europa    1673
Mars      1367
Name: count, dtype: int64

- For non-numerical columns, do dummies.

In [204]:
spaceship["Cabin"].value_counts()

Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64

In [205]:
# Create dummy variables for HomePlanet; Earth is 0, Europa is 1, Mars is 2
planet_map = {'Earth': 0, 'Europa': 1, 'Mars': 2}
spaceship['HomePlanet'] = spaceship['HomePlanet'].map(planet_map)

# Changing CryoSleep to binary values; False is 0, True is 1
spaceship['CryoSleep'] = spaceship['CryoSleep'].astype(int)

# Create dummy variable for Cabin, A is 0, B is 1, C is 2, D is 3, E is 4, F is 5, G is 6, T is 7
cabin_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'T': 7}
spaceship['Cabin'] = spaceship['Cabin'].map(cabin_map)

# Create dummy for destination; TRAPPIST-1e is 0, PSO J318.5-22 is 1, 55 Cancri e is 2
destination_map = {'TRAPPIST-1e': 0, 'PSO J318.5-22': 1, '55 Cancri e': 2}
spaceship['Destination'] = spaceship['Destination'].map(destination_map)

# Changing VIP to binary values; False is 0, True is 1
spaceship['VIP'] = spaceship['VIP'].astype(int)

# Changing Transported to binary values; False is 0, True is 1
spaceship['Transported'] = spaceship['Transported'].astype(int)

spaceship

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,1,0,1,0,39.0,0,0.0,0.0,0.0,0.0,0.0,0
1,0,0,5,0,24.0,0,109.0,9.0,25.0,549.0,44.0,1
2,1,0,0,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0
3,1,0,0,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0
4,0,0,5,0,16.0,0,303.0,70.0,151.0,565.0,2.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
8688,1,0,0,2,41.0,1,0.0,6819.0,0.0,1643.0,74.0,0
8689,0,1,6,1,18.0,0,0.0,0.0,0.0,0.0,0.0,0
8690,0,0,6,0,26.0,0,0.0,0.0,1872.0,1.0,0.0,1
8691,1,0,4,2,32.0,0,0.0,1049.0,0.0,353.0,3235.0,0


**Perform Train Test Split**

In [206]:
features = spaceship.drop(columns = ["Transported"])
target = spaceship["Transported"]

In [207]:
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=0)

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [208]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train, y_train)

KNeighborsClassifier()

- Evaluate your model's performance. Comment it

In [ ]:
# The developed model's accuracy is 77.4%, which is decent for a KNN classifier.
# Additionally, it shows that the feature engineering steps taken were effective in improving the model's performance by around 4%, which is a positive outcome.

knn.score(x_test, y_test)

0.7738275340393344